In [ ]:
import os
import warnings
import re
from datetime import datetime

import pandas as pd

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [ ]:
df_bom = pd.read_excel('bom.xlsx')
df_bom_new = df_bom.copy()
df_bom_new.head()

In [ ]:
def file_generator(path, filename):
    for root, _, files in os.walk(path):
        for file in files:
            if filename.match(file):
                yield file

start_path = "."
target_filename = re.compile(r'^BP.*\.xlsx$')

bp_file_generator = file_generator(start_path, target_filename)
bp_files_done = False

In [ ]:
# При каждом запуске ячейки
if not bp_files_done:
    try:
        next_file = next(bp_file_generator)
        print(next_file)
    except StopIteration:
        bp_files_done = True
        print("Все файлы просмотрены!")
else:
    print("Список файлов закончился. Перезапустите ячейку инициализации для повторного просмотра.")

In [ ]:
bp_number = next_file.split('.')[0]
bp_columns_to_keep = [
    'Change', 'BOM Product', 'Update Type', 'Part No.', 'Part Name(CHN)',
    'Quantity', 'Supplier Name', 'Change Description', 'Solution',
    'Color Code', 'Color Name', 'Production Part Disposal',
    'Interchangeable', 'In Stock', 'New Part Available Date',
    'Workcenter No.', 'Workcenter Name',
]

df_bp = pd.read_excel(f'{bp_number}.xlsx')
df_bp_new = df_bp[bp_columns_to_keep].copy()
df_bp_new['BP_No'] = bp_number
df_bp_new.head()

In [ ]:
df_bp_new.info()

In [ ]:
part_name = df_bp_new['Part Name(CHN)'].unique()
part_name_translation_dict = {}

for name in part_name:
    part_name_translation_dict[name] = ''

part_name_translation_dict

In [ ]:
part_name_translation_dict = {
    '中央电子控制模块': 'Центральный электронный блок управления'
}

df_bp_part_name_translation = df_bp_new.copy()

df_bp_part_name_translation['Part Name (RUS)'] = df_bp_part_name_translation['Part Name(CHN)'].map(part_name_translation_dict)

df_bp_part_name_translation = df_bp_part_name_translation.drop(['Part Name(CHN)'], axis=1)

df_bp_part_name_translation.head()

In [ ]:
supplier_name = df_bp_part_name_translation['Supplier Name'].unique()
supplier_name_translation_dict = {}

for name in supplier_name:
    supplier_name_translation_dict[name] = ''

supplier_name_translation_dict

In [ ]:
supplier_name_translation_dict = {
    '联合汽车电子有限公司': 'United Automotive Electronic Systems Co., Ltd.'
}

df_bp_supplier_name_translation = df_bp_part_name_translation.copy()

df_bp_supplier_name_translation['Supplier Name (RUS)'] = df_bp_supplier_name_translation['Supplier Name'].map(supplier_name_translation_dict).fillna('-')

df_bp_supplier_name_translation = df_bp_supplier_name_translation.drop(['Supplier Name'], axis=1)

df_bp_supplier_name_translation.head()

In [ ]:
def filter_chinese_lines(text):
    """
    Если есть \n - берем строки с китайскими иероглифами
    Если нет \n - берем часть строки до последнего китайского иероглифа
    """
    if not isinstance(text, str):
        return text
    
    chinese_pattern = re.compile(r'[\u4e00-\u9fff]')
    
    # Проверяем, есть ли символ переноса строки
    if '\n' in text:
        # Режим 1: разделяем по \n и берем строки с китайскими иероглифами
        lines = text.split('\n')
        chinese_lines = [line.strip() for line in lines if chinese_pattern.search(line)]
        return ', '.join(chinese_lines) if chinese_lines else text
    else:
        # Режим 2: нет \n - берем часть до последнего китайского иероглифа
        # Находим позицию последнего китайского иероглифа
        last_chinese_pos = -1
        for i, char in enumerate(text):
            if chinese_pattern.match(char):
                last_chinese_pos = i
        
        if last_chinese_pos != -1:
            # Возвращаем часть от начала до последнего иероглифа включительно
            return text[:last_chinese_pos + 1]
        else:
            # Если китайских иероглифов нет, возвращаем исходный текст
            return text

df_bp_description_solution = df_bp_supplier_name_translation.copy()

# Применяем ко всему столбцу, заменяя исходные значения
df_bp_description_solution['Change Description'] = df_bp_description_solution['Change Description'].apply(filter_chinese_lines)
df_bp_description_solution['Solution'] = df_bp_description_solution['Solution'].apply(filter_chinese_lines)

df_bp_description_solution.head()

In [ ]:
change_description = df_bp_description_solution['Change Description'].unique()
change_description_translation_dict = {}

for description in change_description:
    change_description_translation_dict[description] = ''

change_description_translation_dict

In [ ]:
change_description_translation_dict = {
    'B16-R新增本地化CEM，硬件发布': 'Для проекта B16-R добавлен локализованный CEM, выпуск аппаратного обеспечения.'
}

df_bp_description_translation = df_bp_description_solution.copy()

df_bp_description_translation['Change Description (RUS)'] = df_bp_description_translation['Change Description'].map(change_description_translation_dict).fillna('-')

df_bp_description_translation = df_bp_description_translation.drop(['Change Description'], axis=1)

df_bp_description_translation.head()

In [ ]:
solutions = df_bp_description_translation['Solution'].unique()
solution_translation_dict = {}

for solution in solutions:
    solution_translation_dict[solution] = ''

solution_translation_dict

In [ ]:
solution_translation_dict = {
    'B16-R新增本地化CEM，硬件发布, 新增硬件3608101XGW01A，删除硬件3608102XKN02A': 
    'Для проекта B16-R добавлен локализованный CEM, выполнен выпуск аппаратного обеспечения.\nДобавлено аппаратное обеспечение 3608101XGW01A, удалено аппаратное обеспечение 3608102XKN02A.'
}

df_bp_solution_translation = df_bp_description_translation.copy()

df_bp_solution_translation['Solution (RUS)'] = df_bp_solution_translation['Solution'].map(solution_translation_dict).fillna('-')

df_bp_solution_translation = df_bp_solution_translation.drop(['Solution'], axis=1)

df_bp_solution_translation.head()

In [ ]:
color_name = df_bp_solution_translation['Color Name'].unique()
color_translation_dict = {}

for name in color_name:
    color_translation_dict[name] = ''

color_translation_dict

In [ ]:
color_translation_dict = {
    '墨玉黑': 'Нефритово-черный'
}

df_bp_color_translation = df_bp_solution_translation.copy()

df_bp_color_translation['Color Name (RUS)'] = df_bp_color_translation['Color Name'].map(color_translation_dict).fillna('-')

df_bp_color_translation = df_bp_color_translation.drop(['Color Name'], axis=1)

df_bp_color_translation.head()

In [ ]:
def extract_parentheses_content(text):
    # Проверяем на пустые значения
    if pd.isna(text):
        return text
    
    # Преобразуем в строку
    text_str = str(text)
    
    # Ищем ВСЕ тексты в круглых скобках (поддерживает как китайские (), так и английские ())
    matches = re.findall(r'[（(](.*?)[）)]', text_str)
    
    # Если нашли скобки, объединяем все найденные значения через пробел
    if matches:
        return ' '.join(matches)
    else:
        return text_str

df_bp_workcenter_extraction = df_bp_color_translation.copy()

# Применяем функцию к колонке
df_bp_workcenter_extraction['Workcenter Name'] = df_bp_workcenter_extraction['Workcenter Name'].apply(extract_parentheses_content)

df_bp_workcenter_extraction.head()

In [ ]:
workcenter_name = df_bp_solution_translation['Workcenter Name'].unique()
workcenter_translation_dict = {}

for name in workcenter_name:
    workcenter_translation_dict[name] = ''

workcenter_translation_dict

In [ ]:
workcenter_translation_dict = {
    '前围加强板总成W203': 'Сборочная линия усилителя передней панели W203'
}

df_bp_workcenter_extraction['Workcenter Name'] = df_bp_workcenter_extraction['Workcenter Name'].map(workcenter_translation_dict).fillna('-')

df_bp_workcenter_extraction.head()

In [ ]:
df_bp_is_in_bom = df_bp_workcenter_extraction.copy()

# Создаем составной ключ для точного сравнения
df_bp_is_in_bom['Composite Key'] = df_bp_is_in_bom['BOM Product'].astype(str) + '|' + df_bp_is_in_bom['Part No.'].astype(str)
df_bom_new['Composite Key'] = df_bom_new['Model'].astype(str) + '|' + df_bom_new['Part number'].astype(str)

# Проверяем наличие составных ключей
df_bp_is_in_bom['Is in BOM'] = df_bp_is_in_bom['Composite Key'].isin(df_bom_new['Composite Key'])

# Удаляем колонку 'Composite Key' т.к. она больше не нужна
df_bp_is_in_bom = df_bp_is_in_bom.drop(['Composite Key'], axis=1)

df_bp_is_in_bom.head()

In [ ]:
bp_columns_order = [
    'BP_No', 'In Stock', 'New Part Available Date', 'BOM Product', 'Change', 'Update Type',
    'Is in BOM', 'Part No.', 'Part Name (RUS)', 'Workcenter No.', 'Workcenter Name',
    'Quantity', 'Production Part Disposal', 'Interchangeable', 'Supplier Name (RUS)',
    'Change Description (RUS)', 'Solution (RUS)', 'Color Code', 'Color Name (RUS)',
]

df_bp_columns_order = df_bp_is_in_bom.copy()

df_bp_columns_order= df_bp_columns_order[bp_columns_order]
df_bp_columns_order.head()

In [ ]:
def find_file(start_path, target_filename):
    """Ищет файл с указанным именем, начиная с start_path"""
    for root, _, files in os.walk(start_path):
        if target_filename in files:
            return os.path.join(root, target_filename)
    return None

# Получаем текущую дату в формате ГГГГММДД
current_date = datetime.now().strftime('%Y-%m-%d')

directory_name = "."
file_name = f"{current_date}_{bp_number}_refactored.xlsx"

result = find_file(directory_name, file_name)

try:
    if result:
        print(f'Файл с именем {file_name} уже существует по пути {result}')
        file_name_v2 = file_name.split('.')[0]+'_v2.xlsx'
        print(f'Файл сохранен с именем {file_name_v2}')
        df_bp_columns_order.to_excel(
            file_name_v2,
            index=False
        )
    else:
        df_bp_columns_order.to_excel(
            file_name,
            index=False
        )
        print(f'Файл {file_name} сохранен успешно!')
except Exception as e:
    print(f'Во время сохранения файла {file_name} произошла ошибка: {e}')

In [ ]:
# df_bp_list = pd.read_excel('2026-03-31_breakpoint_no.xlsx')
# df_bp_list_new = df_bp_list.copy()
# df_bp_list_new.head()

In [ ]:
# df_bp_list_new.info()

In [ ]:
# df_bp_list_new.columns

In [ ]:
# # 1. Маска для поиска "локал"
# text_mask = (
#     df_bp_list_new['Change Description'].str.contains('локал', case=False, na=False) | 
#     df_bp_list_new['Change Solution'].str.contains('локал', case=False, na=False)
# )

# # 2. Маска для проверки на прочерк
# supplier_mask = df_bp_list_new['Supplier Name Afrter'] == '-'

# # Применяем оба условия (текст найден И в поставщике прочерк)
# unique_bp = df_bp_list_new.loc[text_mask & supplier_mask, 'BP_No'].unique()

# print(unique_bp)

In [ ]:
# directory_name = "."

# for f in unique_bp:
#     f= f+'.xlsx'
#     result = find_file(directory_name, f)
#     if result:
#         print(f'Файл {f} скачан!')
#     else:
#         print(f'Файл {f} нужно скачать!')

In [ ]:
df_bp_BP26003074 = pd.read_excel('2026-04-11_BP26003074_refactored.xlsx')

df_bp_BP26003074.head()

In [ ]:
df_bp_BP26003074_sorted = df_bp_BP26003074.sort_values(
    by=['Update Type', 'Part No.'],
    ascending=[True, True],
    ignore_index=True
)

df_bp_BP26003074.head()

In [ ]:
df_bp_BP26003074.columns

In [ ]:
df_bp_BP26003074['Update Type'].unique()

In [ ]:
part_name_delete_set = set(df_bp_BP26003074[df_bp_BP26003074['Update Type'] == 'Delete']['Part Name (RUS)'].unique())
part_name_delete_set

In [ ]:
part_name_add_set = set(df_bp_BP26003074[df_bp_BP26003074['Update Type'] == 'Add']['Part Name (RUS)'].unique())
part_name_add_set

In [ ]:
diff_del_add = part_name_delete_set - part_name_add_set
diff_del_add

In [ ]:
diff_add_del = part_name_add_set - part_name_delete_set
diff_add_del

In [ ]:
columns = [
    'BP_No', 'Status', 'Batch plan', 'New Part Available Date', 'Batch fact',
    'Change Date', 'BOM Product', 'Part No. Before', 'Part Name Before',
    'Quantity in SS', 'Quantity batches in SS',
    'Configuration for old parts using out',
    'Batches for old parts using out', 'Transmission', 'Part No. After',
    'Part Name After', 'Workcenter No. Before', 'Workcenter Name Before',
    'Workcenter No. After', 'Workcenter Name After',
    'Quantity per Vehicle Before', 'Quantity per Vehicle After',
    'Quantity per Box Before', 'Quantity per Box After',
    'Box Before (L-W-H) mm', 'Box After (L-W-H) mm',
    'Pallet Before (L-W-H) mm', 'Pallet After (L-W-H) mm',
    'Production Part Disposal', 'Interchangeable', 'Supplier Name Before',
    'Localization Before', 'Supplier Name Afrter', 'Localization After',
    'Change Description', 'Change Solution', 'Comments'
]

# Создание пустого датафрейма
df_breakpoint_data = pd.DataFrame(columns=columns)

df_breakpoint_data.head()

In [ ]:
# Сброс индексов у отфильтрованных данных и присвоение по позиции
# part_number_before = df_bp_BP26003074[df_bp_BP26003074['Update Type'] == 'Delete']['Part No.'].reset_index(drop=True)
# sorted_part_number_before = sorted(part_number_before)
# df_breakpoint_data['Part No. Before'] = sorted_part_number_before

part_before = df_bp_BP26003074[df_bp_BP26003074['Update Type'] == 'Delete'][['Part No.', 'Part Name (RUS)']].sort_values('Part No.').reset_index(drop=True)

df_breakpoint_data['Part No. Before'] = part_before['Part No.']
df_breakpoint_data['Part Name Before'] = part_before['Part Name (RUS)']


df_breakpoint_data.head()

In [ ]:
df_breakpoint_data[['Part No. Before', 'Part No. After']]

In [ ]:
df_breakpoint_data = df_breakpoint_data.sort_values(
    by=['Part No. Before', 'Part No. After'],
    ascending=[True, True],
    ignore_index=True
)

df_breakpoint_data[['Part No. Before', 'Part No. After']]